In [1]:
import cadquery as cq
from cadquery import exporters
import math
from jupyter_cadquery import *

from jupyter_cadquery.replay import replay, enable_replay, disable_replay

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

versions()

Overwriting auto display for cadquery Workplane and Shape

Versions:
- jupyter_cadquery  4.0.2
- cad_viewer_widget 3.0.2
- open cascade      7.7.2.1



In [2]:
cv = open_viewer("Joist", cad_width=800, height=600)
set_default_viewer("Joist")

In [3]:
def top_width_from_bottom(bottom_width, depth, angle_deg):
    return bottom_width + 2 * depth * math.tan(math.radians(angle_deg))

def bottom_width_from_top(top_width, depth, angle_deg):
    return top_width - 2 * depth * math.tan(math.radians(angle_deg))

In [4]:
# mm to Inch conversion factor
MM = 1.0 / 25.4

# Sill parameters
sill_length = 48.0
sill_depth = 8.0
sill_height = 10.0

# Joist parameters
joist_length = 72.0
joist_width = 3.0
joist_height = 8.0

# Joint/profile parameters
profile_height = 6.0              # vertical depth down joist face (inches)
bottom_width = 40.0 * MM         # fixed by jig
profile_angle = 4.4926            # trapezoid angle

# Projection into receiving timber
male_insertion_depth = 26.0 * MM
female_insertion_depth = 28.0 * MM

# 15 degree through-thickness dovetail
cutter_angle = 15.0

# Fit
side_clearance = 0.02
vertical_clearance = 0.02


In [5]:
# Derived parameters
top_width = (
    bottom_width
    + 2
    * profile_height
    * math.tan(math.radians(profile_angle))
)

print("bottom:", bottom_width)
print("top:", top_width)
print("height:", profile_height)

bottom: 1.574803149606299
top: 2.517664195246892
height: 6.0


In [6]:
# Female parameters
female_outer_top_width = (
    top_width + 2 * side_clearance
)

female_outer_bottom_width = (
    bottom_width + 2 * side_clearance
)

female_height = (
    profile_height + vertical_clearance
)

female_draft = (
    female_insertion_depth
    * math.tan(math.radians(cutter_angle))
)

female_inner_top_width = (
    female_outer_top_width
    + 2 * female_draft
)

female_inner_bottom_width = (
    female_outer_bottom_width
    + 2 * female_draft
)

In [7]:
def rounded_tail_wire(plane, top_w, bottom_w, height):
    radius = bottom_w / 2.0

    return (
        cq.Workplane(plane)
        .moveTo(-top_w / 2, 0)
        .lineTo(top_w / 2, 0)
        .lineTo(
            bottom_w / 2,
            -height + radius,
        )
        .threePointArc(
            (0, -height),
            (-bottom_w / 2, -height + radius),
        )
        .close()
        .val()
    )

In [8]:
def make_housed_dovetail_tail(
    tail_height,
    tail_depth,
    tail_top_width,
    tail_bottom_width,
    cutter_angle=15.0,
):
    draft = tail_depth * math.tan(math.radians(cutter_angle))

    inner_top_width = tail_top_width + 2 * draft
    inner_bottom_width = tail_bottom_width + 2 * draft

    outer_plane = cq.Plane(
        origin=(0, 0, 0),
        xDir=(1, 0, 0),
        normal=(0, -1, 0),
    )

    inner_plane = cq.Plane(
        origin=(0, tail_depth, 0),
        xDir=(1, 0, 0),
        normal=(0, -1, 0),
    )

    outer_wire = rounded_tail_wire(
        outer_plane,
        tail_top_width,
        tail_bottom_width,
        tail_height,
    )

    inner_wire = rounded_tail_wire(
        inner_plane,
        inner_top_width,
        inner_bottom_width,
        tail_height,
    )

    return cq.Solid.makeLoft(
        [outer_wire, inner_wire],
        ruled=True,
    )

In [9]:
tail = make_housed_dovetail_tail(
    tail_height=profile_height,
    tail_depth=male_insertion_depth,
    tail_top_width=top_width,
    tail_bottom_width=bottom_width,
    cutter_angle=cutter_angle,
)

In [10]:
joist = (
    cq.Workplane("XY")
    .box(
        joist_width,
        joist_length,
        joist_height,
        centered=(True, False, False),
    )
    .translate((0, -joist_length, -joist_height))
)

In [11]:
female_outer_plane = cq.Plane(
    origin=(0, 0, 0),
    xDir=(1, 0, 0),
    normal=(0, -1, 0),
)

female_inner_plane = cq.Plane(
    origin=(0, female_insertion_depth, 0),
    xDir=(1, 0, 0),
    normal=(0, -1, 0),
)

In [12]:
female_outer_wire = rounded_tail_wire(
    female_outer_plane,
    female_outer_top_width,
    female_outer_bottom_width,
    female_height,
)

female_inner_wire = rounded_tail_wire(
    female_inner_plane,
    female_inner_top_width,
    female_inner_bottom_width,
    female_height,
)

In [13]:
female_cutter = cq.Solid.makeLoft(
    [
        female_outer_wire,
        female_inner_wire,
    ],
    ruled=True,
)

In [14]:
sill = (
    cq.Workplane("XY")
    .box(
        sill_length,
        sill_depth,
        sill_height,
        centered=(True, False, False),
    )
    .translate((0, 0, -sill_height))
)

In [15]:
sill_with_socket = sill.cut(female_cutter)
joist_with_tail = joist.union(tail)

In [16]:
show_object(
    sill_with_socket,
    name="sill",
)

show_object(
    joist_with_tail.translate((0, -12, 0)),
    name="joist",
)

+
++
